In [2]:
import sys, ollama
from pathlib import Path
project_root = Path.cwd().parent.parent.parent  # Go up from notebooks/ -> test/ -> src/ -> project root
sys.path.insert(0, str(project_root))

from config.constants import supabase, MODEL_NAME
from src.retrieval import get_document_list, get_document_toc, get_section_content
from src.assessment import generate_questions
from src.evaluation import evaluate_answer
from src.models import check_ollama, generate

print("✅ Imports successful!")
print(f"📦 Model: {MODEL_NAME}\n")

response = ollama.generate(
    model="mistral",
    prompt="Say hello in one word:",
    options={'num_predict': 5}
)

print(f"✅ It works!")
print(f"Response: {response['response']}")

✅ Imports successful!
📦 Model: mistral

✅ It works!
Response:  Hello!


In [3]:
# ============================================
# CELL 3: Full Pipeline Test - FIXED
# ============================================

from src.retrieval import get_document_list, get_document_toc
from src.assessment import generate_questions
from src.evaluation import evaluate_answer

# Get document
docs = get_document_list()

if not docs:
    print("❌ No documents found! Run ingestion first.")
else:
    doc_id = docs[0]['id']
    print(f"📚 Document: {docs[0]['title']}\n")
    
    # Get ToC
    toc = get_document_toc(doc_id)
    
    if not toc:
        print("❌ No ToC found! Check if document was ingested properly.")
    else:
        print(f"✅ Found {len(toc)} chapters\n")
        
        # Find a test section (try to get H2, fall back to H1)
        test_node = None
        for chapter in toc:
            if chapter.get('children') and len(chapter['children']) > 0:
                test_node = chapter['children'][0]
                break
        
        if not test_node:
            test_node = toc[0]
        
        print(f"🎯 Testing section: {test_node['title']}")
        print(f"   Pages: {test_node['page_start']}-{test_node['page_end']}\n")
        
        # Generate questions
        print("⏳ Generating questions... (30-60 seconds)\n")
        result = generate_questions(doc_id, test_node['node_id'], num_questions=3)
        
        if 'error' in result:
            print(f"❌ Error: {result['error']}")
        else:
            print(f"✅ Generated {len(result['questions'])} questions:\n")
            for i, q in enumerate(result['questions'], 1):
                print(f"{i}. {q['question']}\n")
            
            # Evaluate answer
            if result['questions']:
                print("⏳ Evaluating answer... (30-60 seconds)\n")
                test_answer = "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed."
                
                evaluation = evaluate_answer(
                    doc_id,
                    test_node['node_id'],
                    result['questions'][0]['question'],
                    test_answer
                )
                
                if 'error' in evaluation:
                    print(f"❌ Error: {evaluation['error']}")
                else:
                    print(f"📊 Score: {evaluation['score']}/100")
                    if evaluation.get('suggestions'):
                        print(f"💡 Feedback: {evaluation['suggestions']}")

📚 Document: Title Page

❌ No ToC found! Check if document was ingested properly.
